# Notebook 03: Regime Analysis

Market regimes shape which factors work. This notebook covers:
- **Regime state timeline**: when was the market trending, mean-reverting, or volatile?
- **Weight shifts by regime**: how does the adaptive system respond to regime changes?
- **Regime frequency breakdown**: how often does each regime occur?

In [ ]:
import sys
sys.path.append('..')

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path

from src.factors.momentum import MomentumFactor
from src.factors.mean_reversion import MeanReversionFactor
from src.factors.volatility import VolatilityFactor
from src.factors.adaptive_composite import (
    AdaptiveCompositeFactor, AdaptiveCompositeManager,
    RegimeDetector, RegimeAwareCompositeFactor,
)

plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline
print('✅ Imports successful')

## 1. Load Demo Data

In [ ]:
from pathlib import Path

raw_dir = Path('../data/raw')
parquet_files = sorted(raw_dir.glob('*.parquet'))
demo_file = [f for f in parquet_files if '10tickers' in f.name] or [parquet_files[0]]
data = pd.read_parquet(demo_file[0])

print(f'Shape: {data.shape}')
tickers = data.index.get_level_values('ticker').unique().tolist()
print(f'Tickers: {tickers}')

## 2. Compute Market Returns and Detect Regimes

In [ ]:
close_wide = data['close'].unstack('ticker')
mkt_returns = np.log(close_wide / close_wide.shift(1)).mean(axis=1).dropna()
mkt_returns.name = 'market'

detector = RegimeDetector(lookback=63)
dates = mkt_returns.index.tolist()

regimes = []
for date in dates:
    hist = mkt_returns[:date]
    regime = detector.detect_regime(hist)
    regimes.append({'date': date, 'regime': regime})

regime_df = pd.DataFrame(regimes).set_index('date')
print(f'\nRegime counts:\n{regime_df["regime"].value_counts()}')

## 3. Regime State Timeline

In [ ]:
REGIME_COLORS = {
    'trending':      '#2196F3',
    'mean_reverting':'#4CAF50',
    'high_vol':      '#F44336',
    'neutral':       '#9E9E9E',
}

fig, ax = plt.subplots(figsize=(16, 3))
for date, row in regime_df.iterrows():
    color = REGIME_COLORS.get(row['regime'], '#9E9E9E')
    ax.axvspan(date, date + pd.Timedelta(days=1), alpha=0.8, color=color)

patches = [mpatches.Patch(color=c, label=r) for r, c in REGIME_COLORS.items()]
ax.legend(handles=patches, loc='upper right')
ax.set_xlim(regime_df.index[0], regime_df.index[-1])
ax.set_title('Market Regime Timeline')
ax.set_ylabel('Regime')
ax.set_yticks([])
plt.tight_layout()
plt.show()

## 4. Weight Shifts by Regime

In [ ]:
factors = [
    MomentumFactor(lookback=126, name='Momentum'),
    MeanReversionFactor(lookback=21, name='MeanReversion'),
    VolatilityFactor(window=63, name='Volatility'),
]
af = AdaptiveCompositeFactor(factors=factors, ic_window=63, decay_halflife=21, min_weight=0.05)
manager = AdaptiveCompositeManager(af, forward_period=21)

all_dates = sorted(data.index.get_level_values('date').unique())
weight_records = []

for date in all_dates:
    date = pd.Timestamp(date)
    data_slice = data[data.index.get_level_values('date') <= date]
    prices_wide = data['close'].unstack('ticker')
    prices_today = prices_wide[prices_wide.index <= date].iloc[-1]
    try:
        _, weights = manager.compute_factor_with_update(data_slice, prices_today, date)
        regime = regime_df.loc[date, 'regime'] if date in regime_df.index else 'neutral'
        rec = {'date': date, 'regime': regime}
        rec.update(weights)
        weight_records.append(rec)
    except Exception:
        pass

weight_df = pd.DataFrame(weight_records).set_index('date')
factor_cols = [f.name for f in factors]

weight_by_regime = weight_df.groupby('regime')[factor_cols].mean()
print('\nAverage factor weights by regime:')
print(weight_by_regime)

weight_by_regime.plot(kind='bar', figsize=(10, 5))
plt.title('Average Factor Weights by Market Regime')
plt.ylabel('Weight')
plt.xlabel('Regime')
plt.xticks(rotation=0)
plt.legend(title='Factor')
plt.tight_layout()
plt.show()

## 5. Regime Frequency Breakdown

In [ ]:
freq = regime_df['regime'].value_counts()
colors = [REGIME_COLORS.get(r, '#9E9E9E') for r in freq.index]

fig, ax = plt.subplots(figsize=(6, 6))
ax.pie(freq.values, labels=freq.index, colors=colors, autopct='%1.1f%%', startangle=90)
ax.set_title('Regime Frequency Breakdown')
plt.tight_layout()
plt.show()

print(f'\nTotal trading days: {len(regime_df)}')
print(freq.to_string())